In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import os
import random
# import tensorflow_model_optimization as tfmot
from metrics_tracking import F1Score, plot_metrics
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import random
import os
import struct
from keras.models import load_model


# Metrics Before Compression:

Test AUC: 0.8723267912864685 (most important)

Test Precision: 1.0

Test Recall: 0.5199999809265137

Test F1: 0.684210479259491

Test loss: 0.08323828130960464

Test accuracy: 0.9896759390830994

In [3]:
#load dataset for model evaluation
#import the datasets to test



def load_data():
    data = np.load("Preprocessed_Data/roads_canids_windows_200hz_3s.npz")
    # Access arrays by their keys
    X_train = data["X_train"]
    y_train = data["y_train"]
    X_test  = data["X_test"]
    y_test  = data["y_test"]
    X_train = X_train[..., :-1] #TEMP FIX REMOVE LATER - FIX PREPROCESSING TO GET RID OF STRING COLUMN
    X_test  = X_test[..., :-1] #TEMP FIX REMOVE LATER
    print(X_train.shape)
    print(y_train.shape)
    print(X_test.shape)
    print(y_test.shape)
    data.close()
    return X_train, y_train, X_test, y_test

def standardize(X_train, y_train, X_test, y_test):
    # Clip outliers 
    X_train = np.clip(X_train, -1e6, 1e6)
    X_test  = np.clip(X_test,  -1e6, 1e6)
    # Standardize features
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(-1, X_train.shape[-1])
    X_test_flat  = X_test.reshape(-1,  X_test.shape[-1])
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_test_scaled  = scaler.transform(X_test_flat)
    X_train = X_train_scaled.reshape(X_train.shape)
    X_test  = X_test_scaled.reshape(X_test.shape)
    
    print("Scaler mean: ", scaler.mean_, "Scaler scale: ", scaler.scale_) 
    debug_idx = 0
    print("Python X_test_scaled first 10 values:")
    print(X_test_scaled[debug_idx].flatten()[:10])
    return X_train, y_train, X_test, y_test, scaler.mean_, scaler.scale_

X_train, y_train, X_test, y_test = load_data()
X_train, y_train, X_test, y_test, scaler_mean, scaler_std = standardize(X_train, y_train, X_test, y_test)

(11280, 600, 23)
(11280,)
(13948, 600, 23)
(13948,)
Scaler mean:  [2.66699588e+02 2.00110309e+03 2.13149667e+03 1.97537744e+03
 6.56577731e+02 8.49242365e+01 1.57679005e+03 3.84346996e+02
 1.23767696e+02 1.22869378e+01 7.24192116e-01 1.96374107e+00
 5.94536144e+01 1.83361450e+00 2.78406760e-01 1.82680393e+00
 2.19740670e+02 9.06737216e-01 9.48555972e-01 1.70762464e+01
 8.42173483e-01 9.40438800e-03 1.38989805e-01] Scaler scale:  [2.90657690e+02 5.95518380e+03 3.48657453e+04 5.32498429e+03
 3.09433868e+03 2.03155925e+02 6.04737464e+03 5.89790428e+03
 2.42253611e+02 3.72034865e+01 3.57317922e+00 3.30970278e+00
 1.60252947e+02 4.46557535e+00 7.94248569e-01 1.36270320e+00
 7.31552115e+01 9.92689081e-01 2.20900396e-01 3.03348235e+00
 6.69321201e-01 9.64834440e-02 6.30595947e-01]
Python X_test_scaled first 10 values:
[-0.91757279 -0.33602709 -0.06113441 -0.37096399 -0.21218677 -0.41802491
 -0.2607396  -0.06516671 -0.51090135 -0.33026307]


In [ ]:
model = load_model("saved_models/stm32_ROAD_model32_final.keras", compile=True)
y_scores = model.predict(X_test).ravel()
# inspect ranges, use to set threshold post-inferencing for displaying predictions
print("Normal max score: ", y_scores[y_test == 0].max())
print("Attack min score: ", y_scores[y_test == 1].min())

436/436 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Normal max score:  0.50151896
Attack min score:  4.6610618e-05


In [4]:
# SCALE = 1000.0  # float → int16 multiplier
def create_balanced_csv_samples(X_test, y_test, per_class=1):
    attack_idx = np.where(y_test == 1)[0]
    normal_idx = np.where(y_test == 0)[0]
    chosen = random.sample(list(attack_idx), per_class) + \
             random.sample(list(normal_idx), per_class)
    random.shuffle(chosen)
    X_sel = X_test[chosen]
    y_sel = y_test[chosen]
    return X_sel, y_sel

In [ ]:
OUTPUT_DIR = "Preprocessed_Data/STM_SAMPLES"
PREFIX = "samples_fp32"
SAMPLE_COUNT = 2  # number of samples to export

def write_samples_files_fp32(X_sel, y_sel, scaler_mean, scaler_std, prefix="samples_fp32"):
    os.makedirs("Preprocessed_Data/STM_SAMPLES", exist_ok=True)
    h_file = f"Preprocessed_Data/STM_SAMPLES/{prefix}.h"
    c_file = f"Preprocessed_Data/STM_SAMPLES/{prefix}.c"

    S, T, F = X_sel.shape

    # --------- HEADER FILE ---------
    with open(h_file, "w") as h:
        h.write("#pragma once\n")
        h.write("#include <stdint.h>\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n")
        h.write(f"#define SAMPLE_TIME   {T}\n")
        h.write(f"#define SAMPLE_FEATS  {F}\n\n")
        h.write(f"extern const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write(f"extern const int   samples_y[SAMPLE_COUNT];\n")
        h.write(f"extern const float scaler_mean[SAMPLE_FEATS];\n")
        h.write(f"extern const float scaler_std[SAMPLE_FEATS];\n")

    # --------- C FILE ---------
    with open(c_file, "w") as c:
        c.write(f'#include "{prefix}.h"\n\n')
        c.write("const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(f"{v:.6f}f" for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")
        c.write("const int samples_y[SAMPLE_COUNT] = { ")
        c.write(", ".join(str(int(v)) for v in y_sel))
        c.write(" };\n\n")
        c.write("const float scaler_mean[SAMPLE_FEATS] = { ")
        c.write(", ".join(f"{v:.6f}f" for v in scaler_mean))
        c.write(" };\n\n")
        c.write("const float scaler_std[SAMPLE_FEATS] = { ")
        c.write(", ".join(f"{v:.6f}f" for v in scaler_std))
        c.write(" };\n")
    print(f"Generated {h_file} and {c_file} successfully.")


In [ ]:
# X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=1)
# write_samples_files_fp16_safe(X_sel, y_sel, prefix="samples_fp16")
 # Select balanced subset
X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=1)
model = load_model("saved_models/stm32_ROAD_model32_final.keras", compile=True)
print("Attack sample prediction:",
      model.predict(X_sel[y_sel == 1]))
print("Normal sample prediction:",
      model.predict(X_sel[y_sel == 0]))
print(y_sel)


NameError: name 'create_balanced_csv_samples' is not defined

In [ ]:
# n_attack = np.sum(y_train == 1)
# n_normal = np.sum(y_train == 0)

# print(n_normal, n_attack)
# print("Suggested attack weight:", n_normal / n_attack)

10214 1066
Suggested attack weight: 9.581613508442777


In [ ]:
# Generate FP32 .h/.c
write_samples_files_fp32(X_sel, y_sel, scaler_mean, scaler_std, prefix="samples_fp32")

In [ ]:
def write_samples_files_fp32_raw(X_sel, y_sel, prefix="samples_fp32"):
    S, T, F = X_sel.shape

    folder = "Preprocessed_Data/STM_SAMPLES"
    os.makedirs(folder, exist_ok=True)
    with open(os.path.join(folder, f"{prefix}.h"), "w") as h:
        h.write("#pragma once\n")
        h.write("#include <stdint.h>\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n")
        h.write(f"#define SAMPLE_TIME   {T}\n")
        h.write(f"#define SAMPLE_FEATS  {F}\n\n")
        h.write("extern const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write("extern const int samples_y[SAMPLE_COUNT];\n")
    with open(os.path.join(folder, f"{prefix}.c"), "w") as c:
        c.write(f'#include "{prefix}.h"\n')
        c.write("#include <stdint.h>\n\n")
        c.write("const float samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(f"{v:.8f}f" for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")
        c.write("const int samples_y[SAMPLE_COUNT] = { ")
        c.write(", ".join(str(int(v)) for v in y_sel))
        c.write(" };\n")

In [32]:
X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=1)
write_samples_files_fp32_raw(X_sel, y_sel, prefix="samples_fp32_single")

In [5]:

SCALE = 1.0  # FP16 stores raw float values

def float_to_fp16(val):
    """Convert float32 to IEEE 754 half-precision (uint16)."""
    s = struct.pack('e', val)  # half-precision binary
    return struct.unpack('H', s)[0]

def write_samples_files_fp16_from_csv(csv_files, y_csv, prefix="samples"):
    """
    csv_files: list of CSV paths, each containing one sample of shape [T, F]
    y_csv: CSV containing labels (1D array)
    """
    X_list = []
    for f in csv_files:
        X = np.loadtxt(f, delimiter=',', dtype=np.float32)
        X_list.append(X)
    X_sel = np.stack(X_list, axis=0)  # shape: [S, T, F]
    y_sel = np.loadtxt(y_csv, dtype=int)  # shape: [S]

    S, T, F = X_sel.shape
    os.makedirs("Preprocessed_Data", exist_ok=True)

    # Write header
    with open(f"Preprocessed_Data/{prefix}.h", "w") as h:
        h.write("#pragma once\n\n")
        h.write(f"#define SAMPLE_COUNT {S}\n#define SAMPLE_TIME {T}\n#define SAMPLE_FEATS {F}\n#define SAMPLE_SCALE {SCALE}f\n\n")
        h.write("extern const uint16_t samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS];\n")
        h.write("extern const int samples_y[SAMPLE_COUNT];\n")

    # Write C file
    with open(f"Preprocessed_Data/{prefix}.c", "w") as c:
        c.write(f'#include "{prefix}.h"\n\n')
        c.write("const uint16_t samples_X[SAMPLE_COUNT][SAMPLE_TIME][SAMPLE_FEATS] = {\n")
        for s in range(S):
            c.write("  {\n")
            for t in range(T):
                row = ", ".join(str(float_to_fp16(float(v))) for v in X_sel[s, t])
                c.write(f"    {{ {row} }},\n")
            c.write("  },\n")
        c.write("};\n\n")
        c.write("const int samples_y[SAMPLE_COUNT] = { " + ", ".join(str(int(v)) for v in y_sel) + " };\n")

In [6]:
X_sel, y_sel = create_balanced_csv_samples(X_test, y_test, per_class=2)
write_samples_files_fp16_from_csv(X_sel, y_sel, prefix="samples_int16")

TypeError: non-string returned while reading data

In [ ]:
# keras_model = keras.models.load_model( #import model for quantization
#     "best_ROAD_model128_F1Fixed.keras",
#     compile=True,
#     custom_objects={"F1Score": F1Score},
#     safe_mode=False
)
keras_model.summary()

ValueError: File not found: filepath=best_ROAD_model128_F1Fixed.keras. Please ensure the file is an accessible `.keras` zip file.